# Sensitivity analysis: coding of Q7, Q9, Q14 and Q16

On the form, these four questions placed "Neutral" third, after "somewhat". The main analysis scores them 1 to 5 in meaning order (Kruskal-Wallis). Here they are tested in three other ways:

- **Top two answers** versus the rest (chi-square, Cramer's V).
- **Form order:** answer codes 1-5 in the order shown on the form (Kruskal-Wallis, epsilon-squared).
- **Somewhat or higher** versus the rest (chi-square, Cramer's V).

For each coding, the four results replace the main ones in the family of 12 primary tests and the Holm correction is redone, to see whether the conclusions change.

**Reads:** `data/survey_clean.csv`

**Writes to `results/`:** `sensitivity_tests.csv`, `sensitivity_conclusions.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

DATA_FILE = Path("../data/survey_clean.csv")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

SEED = 2026
N_BOOT = 2000
ALPHA = 0.05
GROUPS = ["Patient", "Healthcare professional", "AI developer"]

MAIN_OUTCOMES = [  # (number, label, column, test) - the 12 primary outcomes
    (1,  "Q7 bias concern",                          "bias_concern_code",           "kruskal"),
    (2,  "Q8 privacy is protected",                  "privacy_protection_code",     "kruskal"),
    (3,  "Q9 explanations important",               "explanation_importance_code", "kruskal"),
    (4,  "Q10 inform patients",                      "inform_patients",             "chi2"),
    (5,  "Q11 accountable: AI developers",           "acc_ai_developers",           "chi2"),
    (6,  "Q11 accountable: healthcare professional", "acc_healthcare_professional", "chi2"),
    (7,  "Q11 accountable: hospital or clinic",      "acc_hospital_clinic",         "chi2"),
    (8,  "Q12 trust",                                "trust_code",                  "kruskal"),
    (9,  "Q13 autonomy could be undermined",         "autonomy_undermined_code",    "kruskal"),
    (10, "Q14 guidelines are enough",                "guidelines_confidence_code",  "kruskal"),
    (11, "Q15 comfortable with no human",            "no_human_life_death",         "chi2"),
    (12, "Q16 disparities concern",                  "disparities_concern_code",    "kruskal"),
]
ITEMS = {1: "bias_concern", 3: "explanation_importance", 10: "guidelines_confidence", 12: "disparities_concern"}

df = pd.read_csv(DATA_FILE)
df["group"] = pd.Categorical(df["group"], categories=GROUPS)
print("Participants:", len(df))

Participants: 253


## The 12 main p-values

In [2]:
def chi2_test(var):
    t = pd.crosstab(df[var], df["group"]).reindex(columns=GROUPS, fill_value=0)
    exp = stats.contingency.expected_freq(t.values)
    assert (exp < 5).mean() <= 0.20 and (exp >= 1).all(), "Expected counts too small for chi-square: " + var
    return stats.chi2_contingency(t.values, correction=False)


def kruskal_test(var):
    return stats.kruskal(*[df.loc[df["group"] == g, var] for g in GROUPS])


main = pd.DataFrame([{"outcome_no": num, "outcome": label,
                      "p": float((kruskal_test(var) if test == "kruskal" else chi2_test(var)).pvalue)}
                     for num, label, var, test in MAIN_OUTCOMES])
main

,outcome_no,outcome,p
0,1,Q7 bias concern,0.843051
1,2,Q8 privacy is protected,0.661767
2,3,Q9 explanations important,0.051860
3,4,Q10 inform patients,0.429273
4,5,Q11 accountable: AI developers,0.014573
5,6,Q11 accountable: healthcare professional,0.670447
6,7,Q11 accountable: hospital or clinic,0.142659
7,8,Q12 trust,0.980807
8,9,Q13 autonomy could be undermined,0.964837
9,10,Q14 guidelines are enough,0.350543


## The three other codings (95% bootstrap intervals, resampling within groups)

In [3]:
rng = np.random.default_rng(SEED)


def cramers_v(d, var):
    t = pd.crosstab(d[var], d["group"]).reindex(index=[0, 1], columns=GROUPS, fill_value=0).values
    return float(np.sqrt(stats.chi2_contingency(t, correction=False)[0] / t.sum()))


def epsilon_squared(d, var):
    return float(stats.kruskal(*[d.loc[d["group"] == g, var] for g in GROUPS]).statistic / (len(d) - 1))


def boot(func):
    parts = [df[df["group"] == g] for g in GROUPS]
    return np.percentile([func(pd.concat([p.iloc[rng.integers(0, len(p), len(p))] for p in parts]))
                          for _ in range(N_BOOT)], [2.5, 97.5])


def split_test(num, item, suffix, version):
    var = item + suffix
    res = chi2_test(var)
    lo, hi = boot(lambda d: cramers_v(d, var))
    return {"outcome_no": num, "item": item, "version": version, "test": "Chi-square",
            "statistic": res.statistic, "df": res.dof, "p": res.pvalue,
            "effect_size_name": "Cramer's V", "effect_size": cramers_v(df, var), "ci_low": lo, "ci_high": hi,
            **{"pct_" + g: round(100 * df.loc[df["group"] == g, var].mean(), 1) for g in GROUPS}}


rows = []
for num, item in ITEMS.items():
    rows.append(split_test(num, item, "_strong", "Top two answers"))
    var = item + "_code_form"
    res = kruskal_test(var)
    lo, hi = boot(lambda d: epsilon_squared(d, var))
    rows.append({"outcome_no": num, "item": item, "version": "Form order (1-5)", "test": "Kruskal-Wallis",
                 "statistic": res.statistic, "df": 2, "p": res.pvalue, "effect_size_name": "Epsilon-squared",
                 "effect_size": epsilon_squared(df, var), "ci_low": lo, "ci_high": hi})
    rows.append(split_test(num, item, "_any", "Somewhat or higher"))
sensitivity = pd.DataFrame(rows)
sensitivity.to_csv(RESULTS / "sensitivity_tests.csv", index=False)
sensitivity[["outcome_no", "item", "version", "test", "statistic", "p", "effect_size", "ci_low", "ci_high"]].round(4)

,outcome_no,item,version,test,statistic,p,effect_size,ci_low,ci_high
0,1,bias_concern,Top two answers,Chi-square,0.0142,0.9929,0.0075,0.0148,0.1715
1,1,bias_concern,Form order (1-5),Kruskal-Wallis,0.5570,0.7569,0.0022,0.0003,0.0399
2,1,bias_concern,Somewhat or higher,Chi-square,0.2681,0.8746,0.0326,0.0142,0.1781
3,3,explanation_importance,Top two answers,Chi-square,1.1157,0.5724,0.0664,0.0187,0.2146
4,3,explanation_importance,Form order (1-5),Kruskal-Wallis,7.5338,0.0231,0.0299,0.0049,0.0893
5,3,explanation_importance,Somewhat or higher,Chi-square,0.1706,0.9182,0.0260,0.0152,0.1732
6,10,guidelines_confidence,Top two answers,Chi-square,0.2243,0.8939,0.0298,0.0162,0.1747
7,10,guidelines_confidence,Form order (1-5),Kruskal-Wallis,1.9121,0.3844,0.0076,0.0005,0.0518
8,10,guidelines_confidence,Somewhat or higher,Chi-square,1.8006,0.4064,0.0844,0.0213,0.2164
9,12,disparities_concern,Top two answers,Chi-square,0.5398,0.7634,0.0462,0.0168,0.1904


## Do the conclusions change?

In [4]:
VERSIONS = ["Main (1-5 scale)", "Top two answers", "Form order (1-5)", "Somewhat or higher"]
rows = []
for version in VERSIONS:
    family = main.copy()
    if version != VERSIONS[0]:
        alt = sensitivity[sensitivity["version"] == version].set_index("outcome_no")["p"]
        family.loc[family["outcome_no"].isin(alt.index), "p"] = family["outcome_no"].map(alt)
    family["p_holm"] = multipletests(family["p"], method="holm")[1]
    for _, r in family[family["outcome_no"].isin(ITEMS)].iterrows():
        rows.append({"version": version, "outcome_no": r["outcome_no"], "outcome": r["outcome"], "p": r["p"],
                     "p_holm_in_family_of_12": r["p_holm"], "difference_after_holm": r["p_holm"] < ALPHA})
    rows.append({"version": version, "outcome_no": "all 12", "outcome": "number with a difference after Holm",
                 "p": np.nan, "p_holm_in_family_of_12": np.nan, "difference_after_holm": int((family["p_holm"] < ALPHA).sum())})
conclusions = pd.DataFrame(rows)
conclusions.to_csv(RESULTS / "sensitivity_conclusions.csv", index=False)
conclusions[conclusions["outcome_no"] == "all 12"][["version", "difference_after_holm"]]

,version,difference_after_holm
4,Main (1-5 scale),0
9,Top two answers,0
14,Form order (1-5),0
19,Somewhat or higher,0
